# NYC Flights 2013 - Cross-Table Join Questions


## PART II : 

Standalone notebook containing just the cross-table join analysis.

In [3]:
import pandas as pd
import numpy as np
import nycflights13 as nyc

flights_o = nyc.flights
airlines_o = nyc.airlines
airports_o = nyc.airports
planes_o = nyc.planes
weather_o = nyc.weather



C:\Users\anupr\AppData\Roaming\Python\Python314\site-packages\nycflights13\__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename as _rf


## Setup — feature engineering/ data cleaning

Originally carried out in Part 1 notebook - recreated here so this notebook is also self-contained.

In [4]:
##**Flights**

flights= flights_o.copy()

#convert main datetime-related columns safely
date_cols=["time_hour"]
for col in date_cols:
    flights[col] = pd.to_datetime(flights[col], errors="coerce")

#replace any empty fields with NaN (not a number)
flights["tailnum"]=flights["tailnum"].replace("",np.nan)

#remove exact duplicates
flights = flights.drop_duplicates().reset_index(drop=True) #reset the column numbers to 1,2,3,...

print("Cleaned Shape:",flights.shape)
#print(flights)

###IMP NOTE: NO further FE
#`dep_time`, `dep_delay`, `arr_time`, `arr_delay`, and `air_time` are  missing
#for the same ~8,255 rows — these are cancelled flights, not bad data. 
#`tailnum` is missing for a smaller set (2,512 rows), most likely older/decommissioned aircraft not in FAA records. 

## **Weather**

weather=weather_o.copy()
date_cols = ["time_hour"]
for col in date_cols:
    weather_o[col] = pd.to_datetime(weather_o[col], errors="coerce")
    
#2. clean wind speed outlier value
weather = weather.drop(weather[weather["wind_speed"] > 1000].index)

#3. dropping wind gust column altogether 
weather = weather.drop(columns=["wind_gust"])

#display(weather.head(5))

#print("Cleaned shape:", weather.shape)


###**Planes**
planes= planes_o.copy()
planes = planes.drop(columns=["speed"])

#display(planes.head(5))

#print("Cleaned shape:", planes.shape)

###**Airport**

airports= airports_o.copy()
#display(airport.head(5))

#print("Cleaned shape:", airport.shape)

###**Airlines**
airlines= airlines_o.copy()

#display(airlines.head(5))

#print("Cleaned shape:", airlines.shape)

Cleaned Shape: (336776, 19)


### SET A - FLights and Airlines

### A1. Which airline serves the most distinct destinations from NYC (route)?

In [3]:
dest=flights.groupby("carrier")["dest"].nunique().sort_values(ascending=False)

dest=dest.to_frame("n_destinations").merge(airlines, on="carrier").set_index("name")
print(dest.head(5))

                         carrier  n_destinations
name                                            
ExpressJet Airlines Inc.      EV              61
Endeavor Air Inc.             9E              49
United Air Lines Inc.         UA              47
JetBlue Airways               B6              42
Delta Air Lines Inc.          DL              40


#### Response:
ExpressJet (EV) serves the most distinct destinations (61), well ahead of Endeavor
Air (49) and United (47).

### A2. Which airlines top/dominate the 3 NYC origin airports?

In [5]:
dom=flights.groupby(["origin", "carrier"]).size().reset_index(name="n")

top_per_origin=dom.loc[dom.groupby("origin")["n"].idxmax()].merge(airlines, on="carrier")

print(top_per_origin[["origin", "carrier", "name", "n"]])

  origin carrier                   name      n
0    EWR      UA  United Air Lines Inc.  46087
1    JFK      B6        JetBlue Airways  42076
2    LGA      DL   Delta Air Lines Inc.  23067


#### Response: 
United at Newark (46,087 flights), 
JetBlue at JFK (42,076), and 
Delta at LaGuardia (23,067). 

No single airline dominates across all three.

### A3. Which airline operates the most flights per day on average ?

In [6]:
freq=round((flights.groupby("carrier").size()/365).sort_values(ascending=False))

freq=freq.to_frame("avg_flights_per_day").merge(airlines,on="carrier").set_index("name")

print(freq.head(3))

                         carrier  avg_flights_per_day
name                                                 
United Air Lines Inc.         UA                161.0
JetBlue Airways               B6                150.0
ExpressJet Airlines Inc.      EV                148.0


#### Response 
United Airlines averages 161 flights/day out of NYC, the highest
followed by JetBlue (150) and ExpressJet (148)

### A4. How many airlines serve all 3 NYC origin airports vs. just one?

In [8]:
origins_per_carrier=flights.groupby("carrier")["origin"].nunique().sort_values()

origins_per_carrier=origins_per_carrier.to_frame("n_origins").merge(airlines,on="carrier").set_index("name")

print(origins_per_carrier)

                            carrier  n_origins
name                                          
Alaska Airlines Inc.             AS          1
AirTran Airways Corporation      FL          1
Frontier Airlines Inc.           F9          1
Hawaiian Airlines Inc.           HA          1
Mesa Airlines Inc.               YV          1
Virgin America                   VX          2
SkyWest Airlines Inc.            OO          2
Southwest Airlines Co.           WN          2
Delta Air Lines Inc.             DL          3
ExpressJet Airlines Inc.         EV          3
American Airlines Inc.           AA          3
JetBlue Airways                  B6          3
Endeavor Air Inc.                9E          3
Envoy Air                        MQ          3
United Air Lines Inc.            UA          3
US Airways Inc.                  US          3


#### Response 
8 of 16 airlines serve all three NYC airports: Delta, ExpressJet, American, JetBlue, Endeavor, Envoy, United and US Airways. 
3 airlines serve two airports, and 5 serve only one.

## Set B — Flights and Airports

### B1. Which destination time zones do NYC flights most commonly reach?

In [10]:
fa=flights.merge(airports[["faa", "name", "tzone"]],
                    left_on="dest", right_on="faa", how="left")

print(fa["tzone"].value_counts().head(6))

tzone
America/New_York       192377
America/Chicago         74811
America/Los_Angeles     46324
America/Denver          10291
America/Phoenix          4656
Pacific/Honolulu          707
Name: count, dtype: int64


#### Response
Most flights stay within America/New_York (192,377)

### B2. How many distinct destination time zones does NYC's flight network reach?

In [12]:
print(fa["tzone"].nunique())

7
